# 1. Sharpe ratio inference: usage examples

This notebook shows how to use `stratval.sharpe` to test whether a strategy's Sharpe ratio is greater than zero (Lo, 2002).

We build four synthetic monthly return series of 20 years ($T = 240$), where we know the true Sharpe ratio:

| | true $SR = 0$ | true $SR > 0$ |
|---|---|---|
| **IID normal** | A | B |
| **stationary AR(1)** | C | D |

and run the one-sided test $H_0: SR \leq 0$ against $H_1: SR > 0$ with the three variance methods (`iid_normal`, `iid`, `hac`). A single sample can be lucky or unlucky, so we finish with a small Monte Carlo that measures how often each method rejects.

In [2]:
import sys
from pathlib import Path

# make the stratval package importable when the notebook runs from notebooks/
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np

from stratval.utils import ReturnSeries
from stratval.sharpe.sharpe import sharpe_ratio, sharpe_test

## Synthetic data

Both generators produce returns with mean $\mu$ and standard deviation $\sigma$, so the true per-period Sharpe ratio is $SR = \mu / \sigma$ (risk-free rate $0$).

- **IID normal**: $R_t = \mu + \sigma Z_t$, $Z_t \sim N(0, 1)$ IID.
- **AR(1)**: $R_t = \mu + \varepsilon_t$, $\varepsilon_t = \rho\, \varepsilon_{t-1} + u_t$, $u_t \sim N(0, \sigma^2 (1 - \rho^2))$. The innovation variance is chosen so that $\textnormal{Var}(\varepsilon_t) = \sigma^2$, and $\varepsilon_0$ is drawn from $ N(0, \sigma^2) $, so the series is stationary from the first observation.

In [4]:
def iid_normal_returns(sr, n_obs, sigma, rng):
    return rng.normal(sr * sigma, sigma, size=n_obs)


def ar1_returns(sr, n_obs, sigma, rho, rng):
    u = rng.normal(0.0, sigma * np.sqrt(1 - rho**2), size=n_obs)
    eps = np.empty(n_obs)
    eps[0] = rng.normal(0.0, sigma)  # stationary start
    for t in range(1, n_obs):
        eps[t] = rho * eps[t - 1] + u[t]
    return sr * sigma + eps

In [ ]:
N_OBS = 240          # 20 years of monthly data
SIGMA = 0.04         # 4% monthly volatility
TRUE_SR = 0.2        # monthly, about 0.69 annualized with sqrt(12)
RHO = 0.4            # first-order autocorrelation for the stationary case

rng = np.random.default_rng(2026)
datasets = {
    "A: IID normal, SR = 0":   (iid_normal_returns(0.0, N_OBS, SIGMA, rng), 0.0),
    "B: IID normal, SR = 0.2": (iid_normal_returns(TRUE_SR, N_OBS, SIGMA, rng), TRUE_SR),
    "C: AR(1),      SR = 0":   (ar1_returns(0.0, N_OBS, SIGMA, RHO, rng), 0.0),
    "D: AR(1),      SR = 0.2": (ar1_returns(TRUE_SR, N_OBS, SIGMA, RHO, rng), TRUE_SR),
}
series = {
    name: ReturnSeries(values, periods_per_year=12, name=name)
    for name, (values, _) in datasets.items()
}

## One test on one dataset

`sharpe_test` returns a `SharpeTestResult`. The defaults are `method="hac"`, `alternative="greater"`, `sr0=0` and `alpha=0.05`.

In [10]:
result = sharpe_test(
    rs = series["B: IID normal, SR = 0.2"],
    sr0 = 0.0,
    method = "hac",
    alternative = "greater",
    alpha = 0.05
)
result.to_dict()

{'sr': 0.21030219893548036,
 'sr0': 0.0,
 'se': 0.06337261627559526,
 'stat': 3.3185027113432835,
 'p_value': np.float64(0.00045250721946704653),
 'ci': (np.float64(0.10606352120516359), inf),
 'alpha': 0.05,
 'alternative': 'greater',
 'method': 'hac',
 'lags': 4,
 'n_obs': 240}

## All datasets, all methods

For each dataset we print the estimated SR, and for each method the standard error, the z statistic, the p-value and the lower bound of the one-sided 95% confidence interval.

In [11]:
METHODS = ["iid_normal", "iid", "hac"]

header = f"{'dataset':<26}{'true SR':>8}{'SR_hat':>8}  {'method':<11}{'SE':>7}{'z':>7}{'p-value':>9}{'CI low':>8}  reject?"
print(header)
print("-" * len(header))
for name, rs in series.items():
    true_sr = datasets[name][1]
    for i, method in enumerate(METHODS):
        res = sharpe_test(rs, method=method, alternative="greater")
        left = f"{name:<26}{true_sr:>8.2f}{res.sr:>8.3f}" if i == 0 else " " * 42
        print(f"{left}  {method:<11}{res.se:>7.3f}{res.stat:>7.2f}{res.p_value:>9.4f}"
              f"{res.ci[0]:>8.3f}  {'yes' if res.p_value < 0.05 else 'no'}")
    print()

dataset                    true SR  SR_hat  method          SE      z  p-value  CI low  reject?
-----------------------------------------------------------------------------------------------
A: IID normal, SR = 0         0.00   0.067  iid_normal   0.065   1.03   0.1510  -0.040  no
                                            iid          0.065   1.03   0.1510  -0.040  no
                                            hac          0.071   0.94   0.1747  -0.051  no

B: IID normal, SR = 0.2       0.20   0.210  iid_normal   0.065   3.22   0.0006   0.103  yes
                                            iid          0.065   3.26   0.0006   0.104  yes
                                            hac          0.063   3.32   0.0005   0.106  yes

C: AR(1),      SR = 0         0.00   0.029  iid_normal   0.065   0.44   0.3290  -0.078  no
                                            iid          0.064   0.44   0.3287  -0.077  no
                                            hac          0.090   0.32   0.3

What to look at:

- On the **IID** datasets (A, B) the three methods give almost the same standard error: when the IID assumptions hold, robustness costs little.
- On the **AR(1)** datasets (C, D) the HAC standard error is clearly larger. With $\rho = 0.4$ the long-run variance of returns is $(1 + \rho)/(1 - \rho) \approx 2.33$ times the IID one, so the IID standard errors are too small by a factor of about $\sqrt{2.33} \approx 1.5$ and their p-values are too optimistic.

One sample proves little, though: a single dataset can reject or not by chance. The Monte Carlo below repeats the experiment many times.

## Monte Carlo: how often does each method reject?

For each of the four data-generating processes we simulate `N_SIMS` datasets and record how often $H_0: SR \leq 0$ is rejected at 5%.

- When the true $SR = 0$ (A, C), the rejection rate is the **size** of the test: it should be close to 5%.
- When the true $SR = 0.2$ (B, D), it is the **power**: the higher the better, but only once the size is right.

In [13]:
N_SIMS = 1000
generators = {
    "A: IID normal, SR = 0":   lambda rng: iid_normal_returns(0.0, N_OBS, SIGMA, rng),
    "B: IID normal, SR = 0.2": lambda rng: iid_normal_returns(TRUE_SR, N_OBS, SIGMA, rng),
    "C: AR(1),      SR = 0":   lambda rng: ar1_returns(0.0, N_OBS, SIGMA, RHO, rng),
    "D: AR(1),      SR = 0.2": lambda rng: ar1_returns(TRUE_SR, N_OBS, SIGMA, RHO, rng),
}

rng = np.random.default_rng(7)
print(f"{'dataset':<26}" + "".join(f"{m:>12}" for m in METHODS))
for name, generate in generators.items():
    rejections = dict.fromkeys(METHODS, 0)
    for _ in range(N_SIMS):
        rs = ReturnSeries(generate(rng), periods_per_year=12)
        for method in METHODS:
            rejections[method] += sharpe_test(rs, method=method).p_value < 0.05
    print(f"{name:<26}" + "".join(f"{rejections[m] / N_SIMS:>12.1%}" for m in METHODS))

dataset                     iid_normal         iid         hac
A: IID normal, SR = 0             5.0%        4.9%        5.6%
B: IID normal, SR = 0.2          92.0%       92.1%       92.5%
C: AR(1),      SR = 0            14.2%       14.2%        8.1%
D: AR(1),      SR = 0.2          82.7%       82.6%       69.0%


Reading the table:

- **A (IID, SR = 0)**: all methods reject about 5% of the time, as they should.
- **C (AR(1), SR = 0)**: the IID methods reject far more than 5%. They declare a skill-less strategy significant much too often, because they ignore the autocorrelation. HAC is much closer to 5% (usually slightly above: Newey–West with few lags is biased downward in small samples).
- **B, D (SR = 0.2)**: power. In D the IID methods look more powerful, but that is not a real advantage, because their size is wrong.

This is the practical message of Lo (2002): with serially correlated returns, use the HAC standard error.